In [1]:
import re
import math
from collections import Counter

# Sample document collection
documents = {
    1: "Information retrieval is the process of finding relevant information",
    2: "Search engines use information retrieval models to rank documents",
    3: "Inverted indexes improve fast query evaluation in information retrieval",
    4: "Boolean retrieval uses AND OR and NOT operators for searching",
    5: "TF IDF and BM25 are popular ranking models in information retrieval",
    6: "Index compression reduces storage and speeds up query processing",
    7: "Variable byte encoding and gap encoding compress inverted indexes",
    8: "Search systems use indexes to retrieve relevant documents quickly"
}

def tokenize(text):
    return re.findall(r"\b[a-z0-9]+\b", text.lower())

term_frequencies = {
    doc_id: Counter(tokenize(text))
    for doc_id, text in documents.items()
}

def boolean_search(query):
    """Simple left-to-right Boolean evaluation supporting AND, OR and NOT."""
    tokens = re.findall(
        r"\bAND\b|\bOR\b|\bNOT\b|[a-zA-Z0-9]+",
        query,
        flags=re.IGNORECASE
    )

    all_docs = set(documents)
    result = None
    operator = "AND"
    negate_next = False

    for token in tokens:
        upper = token.upper()

        if upper in ("AND", "OR"):
            operator = upper
            continue

        if upper == "NOT":
            negate_next = True
            continue

        matching = {
            doc_id for doc_id, text in documents.items()
            if token.lower() in tokenize(text)
        }

        if negate_next:
            matching = all_docs - matching
            negate_next = False

        if result is None:
            result = matching
        elif operator == "AND":
            result &= matching
        else:
            result |= matching

    return result if result is not None else set()

def idf(term):
    df = sum(
        1 for tf in term_frequencies.values()
        if tf.get(term, 0) > 0
    )
    return math.log((len(documents) + 1) / (df + 1)) + 1

def tfidf_search(query):
    """Cosine similarity using TF-IDF weights."""
    query_tf = Counter(tokenize(query))

    query_weights = {
        term: frequency * idf(term)
        for term, frequency in query_tf.items()
    }

    query_norm = math.sqrt(
        sum(weight ** 2 for weight in query_weights.values())
    )

    scores = {}

    for doc_id, tf in term_frequencies.items():
        weights = {
            term: frequency * idf(term)
            for term, frequency in tf.items()
        }

        doc_norm = math.sqrt(
            sum(weight ** 2 for weight in weights.values())
        )

        dot = sum(
            weights.get(term, 0) * weight
            for term, weight in query_weights.items()
        )

        scores[doc_id] = (
            dot / (doc_norm * query_norm)
            if doc_norm and query_norm else 0.0
        )

    return sorted(
        scores.items(),
        key=lambda item: item[1],
        reverse=True
    )

def show_rankings(title, rankings):
    print("\n" + title)
    print("-" * 65)

    for doc_id, score in rankings:
        if score > 0:
            print(f"Document {doc_id} | Score: {score:.4f}")
            print(documents[doc_id])

if __name__ == "__main__":
    query = "information retrieval"

    print("INFORMATION RETRIEVAL MODELS")
    print("Query:", query)

    print("\nBoolean Model (information AND retrieval):")
    print(sorted(boolean_search("information AND retrieval")))

    show_rankings("TF-IDF MODEL", tfidf_search(query))

    print("\nCOMPARISON")
    print("Boolean: retrieves documents satisfying logical conditions.")
    print("TF-IDF: ranks documents using term importance and cosine similarity.")


INFORMATION RETRIEVAL MODELS
Query: information retrieval

Boolean Model (information AND retrieval):
[1, 2, 3, 5]

TF-IDF MODEL
-----------------------------------------------------------------
Document 1 | Score: 0.4786
Information retrieval is the process of finding relevant information
Document 2 | Score: 0.3393
Search engines use information retrieval models to rank documents
Document 3 | Score: 0.3361
Inverted indexes improve fast query evaluation in information retrieval
Document 5 | Score: 0.2900
TF IDF and BM25 are popular ranking models in information retrieval
Document 4 | Score: 0.1245
Boolean retrieval uses AND OR and NOT operators for searching

COMPARISON
Boolean: retrieves documents satisfying logical conditions.
TF-IDF: ranks documents using term importance and cosine similarity.
